<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>

## **18. Build Your Own Model**

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 18.1 Introduction

This is the project of the chapter. You build a language model from scratch, train it, look inside it, and make it small enough for a device that can hold only **8 KB** of weights, about what a sensor's chip has to spare. Every step is one you have met before, so this notebook explains little and asks you to decide a lot.

| Step | Decision | From |
|:-----|:---------|:-----|
| Data | your sentences, and the ones you keep back for testing | [05.06](05.06_Understanding_Training.ipynb) |
| Tokens | words or characters | [05.01](05.01_Understanding_Tokenization.ipynb) |
| Model | width, heads, layers, context | [05.02](05.02_Understanding_Word_Embedding.ipynb), [05.03](05.03_Understanding_Attention.ipynb) |
| Training | steps and learning rate | [05.06](05.06_Understanding_Training.ipynb) |
| Generating | temperature and top-k | [05.05](05.05_Introduction_to_LLMs.ipynb), [05.12](05.12_Understanding_Inference_Control.ipynb) |
| Shipping | bits per weight, within the budget | [05.07](05.07_Introduction_to_Local_LLMs.ipynb), [05.09](05.09_Making_Models_Smaller.ipynb) |

The model is a GPT in plain Python, adapted from Andrej Karpathy's `microgpt` [01]. Every weight is a Python number you can print. That makes it slow, so it stays tiny: with the default settings, training takes one to two minutes on a laptop.

Allow about **90 minutes**, plus time for the questions.

By the end, you should be able to build and train a small language model yourself, tell from a test set whether it learned or memorised, and defend every design decision with your own numbers.

#### 18.1.1 Environment Setup

Everything runs on your laptop; the Pi is not needed.

<div class="alert alert-block alert-success">

If a library is missing, use the installation instructions in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb). Select that Python environment as this notebook's kernel.
</div>

In [ ]:
import math
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np

### 18.2 Your Decisions

All decisions sit in one place. Run the notebook once with the defaults, then come back here for your own design.

In [ ]:
CONFIG = {
    "tokens": "words",       # "words" or "characters"; characters need a block_size of about 48
    "n_layer": 1,            # transformer layers
    "n_embd": 16,            # width of every token vector
    "n_head": 4,             # attention heads, must divide n_embd
    "block_size": 16,        # context: how many tokens the model sees at once
    "num_steps": 300,        # training steps, one sentence each
    "learning_rate": 0.01,
    "temperature": 0.6,      # for the samples in Section 18.7
    "top_k": 5,
    "bits": 32,              # bits per weight on the device: 32, 8 or 4
    "seed": 2,
}
BUDGET = 8 * 1024            # bytes of weights the device can hold

### 18.3 Data and Tokens

The default data are thirty sentences about baking. The test sentences are kept back: the model never trains on them, so they show whether it learned the patterns or only the sentences. You may replace both lists with a small domain of your own: about thirty short training sentences, and five test sentences that use the same words in new combinations.

In [ ]:
TRAIN = [
    "mix flour and sugar for a soft cake", "mix flour and sugar for sweet cookies",
    "add butter and milk to the cake bowl", "add butter and milk to the cookie bowl",
    "stir the cake batter until it is smooth", "stir the cookie dough until it is soft",
    "bake the cake until the top is golden", "bake the cookies until the edges are golden",
    "cool the cake before adding sweet cream", "cool the cookies before adding sugar dust",
    "slice the cake and serve it warm", "share the cookies and serve them warm",
    "vanilla makes the cake smell sweet", "vanilla makes the cookies smell sweet",
    "chocolate chips make the cookies rich", "chocolate cream makes the cake rich",
    "use eggs to help the cake rise", "use eggs to help the cookies hold shape",
    "a small cake needs flour sugar butter and eggs", "small cookies need flour sugar butter and eggs",
    "the cake pan goes into the oven", "the cookie tray goes into the oven",
    "the oven makes the cake warm and golden", "the oven makes the cookies warm and golden",
    "soft cake tastes good with cream", "soft cookies taste good with milk",
    "sweet cake and sweet cookies are simple treats", "fresh cake and fresh cookies smell good",
    "let the cake rest before you slice it", "let the cookies rest before you share them",
]
TEST = [
    "mix flour and sugar for soft cookies",
    "bake the cookies until the top is golden",
    "add butter and eggs to the cake bowl",
    "the oven makes the cookies smell sweet",
    "let the cake cool before you slice it",
    "mix flour and honey for a soft cake",
]

The tokenizer splits a sentence into words or into single characters (05.01). One extra token, `BOS`, marks the start and the end of every sentence.

In [ ]:
def split(text):
    """The tokens of a text: words or single characters, depending on CONFIG."""
    if CONFIG["tokens"] == "words":
        return re.findall(r"[a-z]+", text.lower())
    return list(text.lower())


vocab = sorted({token for sentence in TRAIN for token in split(sentence)})
token_id = {token: i for i, token in enumerate(vocab)}
BOS = len(vocab)
V = len(vocab) + 1


def encode(text):
    return [token_id[token] for token in split(text)]


def decode(ids):
    return ("" if CONFIG["tokens"] == "characters" else " ").join(vocab[i] for i in ids)


print(f"vocabulary: {V} tokens, BOS included")
print(f"{TRAIN[0]!r} -> {encode(TRAIN[0])}")
for sentence in TEST:
    unknown = [token for token in split(sentence) if token not in token_id]
    if unknown:
        print(f"cannot encode the test sentence {sentence!r}: {unknown} never appears in TRAIN")

With word tokens, one test sentence cannot be encoded: `honey` never appears in `TRAIN`. That is on purpose, and Question 02 comes back to it.

### 18.4 The Model and Its Size

Every weight is a `Value`: a number that remembers how it was computed, so that training can send the gradient back through it (05.06). This class is Karpathy's micrograd idea in thirty lines [01].

In [ ]:
class Value:
    __slots__ = ("data", "grad", "_children", "_local_grads")

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, exponent):
        return Value(self.data ** exponent, (self,), (exponent * self.data ** (exponent - 1),))

    def log(self):
        return Value(math.log(self.data), (self,), (1 / self.data,))

    def exp(self):
        return Value(math.exp(self.data), (self,), (math.exp(self.data),))

    def relu(self):
        return Value(max(0, self.data), (self,), (float(self.data > 0),))

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    def backward(self):
        """Send the gradient from this value back to every value it was computed from."""
        order, seen = [], set()

        def visit(value):
            if value not in seen:
                seen.add(value)
                for child in value._children:
                    visit(child)
                order.append(value)

        visit(self)
        self.grad = 1
        for value in reversed(order):
            for child, local_grad in zip(value._children, value._local_grads):
                child.grad += local_grad * value.grad

The weights are a handful of matrices (05.02, 05.03): token and position embeddings, the attention and MLP matrices of each layer, and the output layer that scores every token of the vocabulary.

In [ ]:
random.seed(CONFIG["seed"])
d, n_layer, n_head = CONFIG["n_embd"], CONFIG["n_layer"], CONFIG["n_head"]
T, head_dim = CONFIG["block_size"], CONFIG["n_embd"] // CONFIG["n_head"]


def matrix(rows, cols):
    return [[Value(random.gauss(0, 0.08)) for _ in range(cols)] for _ in range(rows)]


weights = {"wte": matrix(V, d), "wpe": matrix(T, d), "lm_head": matrix(V, d)}
for layer in range(n_layer):
    for name in ("attn_wq", "attn_wk", "attn_wv", "attn_wo"):
        weights[f"layer{layer}.{name}"] = matrix(d, d)
    weights[f"layer{layer}.mlp_fc1"] = matrix(4 * d, d)
    weights[f"layer{layer}.mlp_fc2"] = matrix(d, 4 * d)
params = [p for m in weights.values() for row in m for p in row]

for name, m in weights.items():
    print(f"{name:16s} {len(m):3d} x {len(m[0]):3d} = {len(m) * len(m[0]):5d}")
print(f"{len(params)} weights in total")

How many bytes is that on the device? Below 32 bits, every row of a matrix gets its own 16-bit scale, the grouping that made 4-bit quantization work in 05.09.

In [ ]:
def stored_bytes(bits):
    """The weights at `bits` each, plus one 16-bit scale per matrix row below 32 bits."""
    if bits == 32:
        return 4 * len(params)
    rows = sum(len(m) for m in weights.values())
    return math.ceil(len(params) * bits / 8) + 2 * rows


for bits in (32, 8, 4):
    size = stored_bytes(bits)
    print(f"{bits:2d} bits: {size:6d} bytes, {'fits' if size <= BUDGET else 'too big'}")

cache = 2 * n_layer * d * T * 4
print(f"\nkeys and values for a full context of {T} tokens, at 32 bits: {cache} bytes")

### 18.5 Training

The forward pass is the same as in every model of this chapter, only small: embeddings, then attention and an MLP per layer, then a score for every token (05.03, 05.05). The loss of a sentence is the average surprise of each next token. It is measured in nats, with the natural logarithm instead of the base-2 logarithm of 05.06, so a clueless model starts near ln V instead of log₂ V.

In [ ]:
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(row, x)) for row in w]


def softmax(logits):
    top = max(value.data for value in logits)
    exps = [(value - top).exp() for value in logits]
    total = sum(exps)
    return [e / total for e in exps]


def rmsnorm(x):
    scale = (sum(xi * xi for xi in x) / len(x) + 1e-5) ** -0.5
    return [xi * scale for xi in x]


def forward(weights, token, position, keys, values):
    """Scores for the next token, given one token, its position and the keys and values so far."""
    x = rmsnorm([t + p for t, p in zip(weights["wte"][token], weights["wpe"][position])])
    for layer in range(n_layer):
        residual = x
        x = rmsnorm(x)
        q = linear(x, weights[f"layer{layer}.attn_wq"])
        keys[layer].append(linear(x, weights[f"layer{layer}.attn_wk"]))
        values[layer].append(linear(x, weights[f"layer{layer}.attn_wv"]))
        heads = []
        for h in range(0, d, head_dim):
            scores = [sum(q[h + j] * k[h + j] for j in range(head_dim)) / head_dim ** 0.5 for k in keys[layer]]
            attention = softmax(scores)
            heads += [sum(a * v[h + j] for a, v in zip(attention, values[layer])) for j in range(head_dim)]
        x = [a + b for a, b in zip(linear(heads, weights[f"layer{layer}.attn_wo"]), residual)]
        residual = x
        x = rmsnorm(x)
        hidden = [xi.relu() for xi in linear(x, weights[f"layer{layer}.mlp_fc1"])]
        x = [a + b for a, b in zip(linear(hidden, weights[f"layer{layer}.mlp_fc2"]), residual)]
    return linear(x, weights["lm_head"])


def sentence_loss(weights, sentence):
    """Average surprise per token over one sentence."""
    tokens = [BOS] + encode(sentence) + [BOS]
    n = min(T, len(tokens) - 1)
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for i in range(n):
        probs = softmax(forward(weights, tokens[i], i, keys, values))
        losses.append(-probs[tokens[i + 1]].log())
    return sum(losses) * (1 / n)


def test_loss(weights):
    """Average loss over the test sentences the tokenizer can encode."""
    known = [s for s in TEST if all(token in token_id for token in split(s))]
    return sum(sentence_loss(weights, s).data for s in known) / len(known)


print(f"ln V = {math.log(V):.2f}, test loss before training: {test_loss(weights):.2f}")

Before training, look at the weights: every matrix is faint noise around zero, on a scale from −1 (blue) to +1 (red).

In [ ]:
def show_weights(weights, title):
    columns = 6
    rows = math.ceil(len(weights) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(13, 2.6 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (name, m) in zip(axes.flat, weights.items()):
        ax.imshow([[p.data for p in row] for row in m], cmap="bwr", vmin=-1, vmax=1, aspect="auto")
        ax.set_title(name, fontsize=8)
    fig.suptitle(title, x=0.01, ha="left")
    plt.show()


show_weights(weights, "before training")

The training loop is the one from 05.06, with the Adam optimizer [02] and a learning rate that shrinks to zero over the run. It prints its progress every 50 steps.

In [ ]:
order = TRAIN[:]
random.Random(CONFIG["seed"]).shuffle(order)
m_adam, v_adam = [0.0] * len(params), [0.0] * len(params)
losses, start = [], time.time()

for step in range(CONFIG["num_steps"]):
    loss = sentence_loss(weights, order[step % len(order)])
    loss.backward()
    lr = CONFIG["learning_rate"] * (1 - step / CONFIG["num_steps"])
    for i, p in enumerate(params):
        m_adam[i] = 0.85 * m_adam[i] + 0.15 * p.grad
        v_adam[i] = 0.99 * v_adam[i] + 0.01 * p.grad ** 2
        m_hat = m_adam[i] / (1 - 0.85 ** (step + 1))
        v_hat = v_adam[i] / (1 - 0.99 ** (step + 1))
        p.data -= lr * m_hat / (v_hat ** 0.5 + 1e-8)
        p.grad = 0
    losses.append(loss.data)
    if step % 50 == 0 or step == CONFIG["num_steps"] - 1:
        print(f"step {step + 1:4d}  loss {loss.data:.2f}  ({time.time() - start:.0f} s)")

train_seconds = time.time() - start

In [ ]:
final_test = test_loss(weights)
plt.figure(figsize=(10, 3.8))
plt.plot(losses, color="#1b9e77", alpha=0.3)
window = min(20, len(losses))   # average over 20 steps, fewer in a very short run
plt.plot(range(window - 1, len(losses)), np.convolve(losses, np.ones(window) / window, mode="valid"),
         color="#1b9e77", label="training loss")
plt.axhline(math.log(V), color="#757575", linestyle="--", label="clueless model: ln V")
plt.axhline(final_test, color="#d95f02", label="test loss after training")
plt.xlabel("training step")
plt.ylabel("loss (nats per token)")
plt.legend()
plt.show()
print(f"training loss over the last 30 steps: {np.mean(losses[-30:]):.2f}, test loss: {final_test:.2f}")

The gap between the two losses is the difference between knowing the training sentences and knowing the patterns behind them.

### 18.6 Inside the Model

The same matrices after training, on the same colour scale:

In [ ]:
show_weights(weights, "after training")

The token embeddings, projected onto their two main directions as in 05.02. Tokens that play the same role in the sentences should end up close together.

In [ ]:
E = np.array([[p.data for p in row] for row in weights["wte"][:BOS]])
E = E - E.mean(axis=0)
directions = np.linalg.svd(E, full_matrices=False)[2][:2]
xy = E @ directions.T

plt.figure(figsize=(8, 6))
plt.scatter(xy[:, 0], xy[:, 1], s=12, color="#4c78a8")
for (x, y), token in zip(xy, vocab):
    plt.annotate(token if token != " " else "space", (x, y), fontsize=8)
plt.title("token embeddings, first two main directions", loc="left")
plt.show()

### 18.7 Generating Text

`complete` writes one token at a time, as in 05.05: it keeps the `top_k` most likely tokens, reshapes them with the temperature and draws one, until it draws `BOS` or the context is full.

In [ ]:
def complete(prompt, weights=weights, temperature=CONFIG["temperature"], top_k=CONFIG["top_k"], seed=0):
    rng = random.Random(seed)
    tokens = [BOS] + encode(prompt)
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    for position, token in enumerate(tokens[:-1]):
        forward(weights, token, position, keys, values)
    new, token = [], tokens[-1]
    for position in range(len(tokens) - 1, T):
        scores = [s.data for s in forward(weights, token, position, keys, values)]
        best = sorted(range(V), key=lambda i: -scores[i])[:top_k]
        top = scores[best[0]]
        token = rng.choices(best, [math.exp((scores[i] - top) / temperature) for i in best])[0]
        if token == BOS:
            break
        new.append(token)
    return decode(new)


PROMPTS = ["mix flour and", "the oven makes", "let the"]
for prompt in PROMPTS:
    for seed in (0, 1):
        print(f"{prompt} -> {complete(prompt, seed=seed)}")

Read each sample against `TRAIN`. Is it a training sentence, memorised? A new mix of training sentences, recombined? Or something that appears nowhere, invented? Invented text is what a large model does when it hallucinates (05.13): fluent, plausible and not backed by anything it saw.

Try a prompt of your own and other temperatures. A prompt may only use tokens from the vocabulary.

In [ ]:
print(complete("bake the", temperature=0.2))
print(complete("bake the", temperature=1.5, top_k=V))

### 18.8 Shipping to the Device

The device holds 8 KB. `rounded` stores every weight with fewer bits, one scale per matrix row, the group-wise quantization of 05.09. Rounding happens after training, so you can try every bit width without training again.

In [ ]:
def rounded(weights, bits):
    """A copy of the weights with `bits` per weight and one scale per matrix row."""
    if bits == 32:
        return weights
    levels = 2 ** (bits - 1) - 1
    copy = {}
    for name, m in weights.items():
        copy[name] = []
        for row in m:
            scale = max(abs(p.data) for p in row) / levels
            if scale == 0:
                scale = 1.0
            copy[name].append([Value(round(p.data / scale) * scale) for p in row])
    return copy


print(f"{'bits':>4s} {'bytes':>7s} {'fits':>5s} {'test loss':>10s}")
for bits in (32, 8, 4, 3, 2):
    size = stored_bytes(bits)
    print(f"{bits:4d} {size:7d} {'yes' if size <= BUDGET else 'no':>5s} {test_loss(rounded(weights, bits)):10.2f}")

Choose `bits` in Section 18.2 and run this section again. The next cell shows what ships: the size, the test loss, and the same prompts as before, written by the rounded model.

In [ ]:
shipped = rounded(weights, CONFIG["bits"])
size = stored_bytes(CONFIG["bits"])
print(f"{CONFIG['bits']} bits: {size} bytes, {'fits into' if size <= BUDGET else 'too big for'} {BUDGET} bytes")
print(f"test loss {test_loss(shipped):.2f} (unrounded {final_test:.2f})\n")
for prompt in PROMPTS:
    print(f"{prompt} -> {complete(prompt, weights=shipped)}")

### 18.9 Your Design

<div class="alert alert-block alert-info">

Find the best model that fits into 8 KB. Change the settings in Section 18.2, run the notebook again from the top (Restart Kernel, then Run All), and compare. Change one setting at a time, so that you know what each change did. Use the test loss as your measure, not the look of a single sample. One exception: a word model's loss is the surprise per word, a character model's the surprise per character. To compare the two, multiply each test loss by the average number of tokens in a test sentence: `np.mean([len(split(s)) + 1 for s in TEST])`, where the `+ 1` counts the end of the sentence. Keep in mind the time: a wider or deeper model trains slower, and characters make every sentence about five times longer.
</div>

The next cell prints one line per run. Copy it into Question 09 after every run that you want to compare.

In [ ]:
print(f"{CONFIG['tokens']}, {n_layer} layer(s), width {d}, {n_head} heads, context {T}, {len(params)} weights, "
      f"{CONFIG['num_steps']} steps in {train_seconds:.0f} s | train loss {np.mean(losses[-30:]):.2f}, "
      f"test loss {final_test:.2f} | {CONFIG['bits']} bits: {stored_bytes(CONFIG['bits'])} bytes, "
      f"test loss {test_loss(shipped):.2f}")

### 18.10 What Your Model Is Not

Your model works like the models on the Pi, which are a hundred thousand to a million times larger. It is also missing everything that came after training in this chapter. It continues text but does not answer questions, because it never had instruction tuning (05.06). It knows only its sentences and cannot look anything up (05.10). It cannot call tools or act in a loop (05.15 to 05.17). And it has no idea when it does not know (05.19). The last question asks you to put this into words.

### 18.11 Further Reading and Literature

[01] Karpathy, A. **microgpt** and **micrograd**. The model and the `Value` class of this notebook are adapted from them. [micrograd](https://github.com/karpathy/micrograd), [gists](https://gist.github.com/karpathy)

[02] Kingma, D. P., & Ba, J. (2015). **Adam: A Method for Stochastic Optimization.** ICLR. [Paper](https://arxiv.org/abs/1412.6980)

[03] Karpathy, A. [**nanoGPT**](https://github.com/karpathy/nanoGPT). The next step up: the same model in PyTorch, large enough to train on real text.

---
### 18.12 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own outputs as evidence. Copy the relevant numbers into your answer so that they remain visible after saving.

**Question 01: Your decisions**

Report your final settings from Section 18.2. For every setting you changed, say why.

(answer here)

**Question 02: Words or characters**

Compare the two tokenizers: the vocabulary size, the tokens per sentence, and what happened with the test sentence about honey. Why can you not compare their test losses directly? Which tokenizer would you choose for the device, and why?

(answer here)

**Question 03: Where the bytes go**

Which matrix holds the most weights, and why does its size depend on the vocabulary? Report the bytes at 32, 8 and 4 bits. How do the keys and values of a full context compare with the budget?

(answer here)

**Question 04: The first loss**

Report the test loss before training and ln V. Why must they be close?

(answer here)

**Question 05: Learned or memorised**

Report the training and the test loss after training. What does the gap between them tell you, and how would more training steps change it?

(answer here)

**Question 06: Inside the model**

Describe one matrix that changed visibly during training, and one group of tokens that ended up close together in the embedding plot. Why would training move them together?

(answer here)

**Question 07: Your samples**

Give one memorised, one recombined and one invented sample, each with the training sentences behind it. What did temperature and top-k change?

(answer here)

**Question 08: Shipping**

From the bits table in Section 18.8: where did the test loss start to grow, and where did it collapse? Is a larger model at 4 bits better than the default model at 8 bits? Use your numbers.

(answer here)

**Question 09: Your design**

Paste the lines of Section 18.9 for the default and for your best model that fits into 8 KB. Explain why your changes helped, or why they did not.

(answer here)

**Question 10: What your model is not**

Pick two of the notebooks 05.10 to 05.17 and 05.19. For each, explain what your model would need in order to do what you did there, and which of the techniques from this chapter would get it there.

(answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)